In [ ]:
# --- Config: paths & helpers (auto-inserted) ---
from pathlib import Path
import os, json
import pandas as pd

# Ensure we're in the repository root (where logs directories are)
REPO = Path.cwd()
if REPO.name == "notebooks":
    REPO = REPO.parent
    os.chdir(str(REPO))
    print(f"Changed working directory to repository root: {REPO}")

os.environ["PYTHONPATH"] = str(REPO)

# Core paths
META = REPO / "meta" / "master_metadata.csv"
LOGS = REPO / "logs"
LOGS_EF = REPO / "logs_ef"
LOGS_VOL = REPO / "logs_vol"
REPORTS = REPO / "reports"
REPORTS_SEG = REPO / "reports_seg"
REPORTS.mkdir(exist_ok=True); REPORTS_SEG.mkdir(exist_ok=True)

# --- Loaders ---
def load_seg_summaries():
    out = {}
    if LOGS.exists():
        for p in LOGS.glob("cv_seg_*_summary.json"):
            try:
                with open(p) as f:
                    out[p.stem] = json.load(f)
            except Exception:
                pass
    return out

def load_clf_summaries():
    out = {}
    p = LOGS / "cv_cls_summary.csv"
    if p.exists():
        out["all"] = pd.read_csv(p)
    p = LOGS_EF / "cv_cls_summary.csv"
    if p.exists():
        out["ef"] = pd.read_csv(p)
    p = LOGS_VOL / "cv_cls_summary.csv"
    if p.exists():
        out["vol"] = pd.read_csv(p)
    return out

print("Pipeline notebook configuration complete.")

# Cardiac Early Detection — Pipeline Notebook
Use this notebook to run the dataset processing and CV/HPO end-to-end.

## 🚀 Quick Start

This notebook provides an end-to-end pipeline for cardiac disease classification and segmentation research. 

### ⚡ **Prerequisites**
- Raw CAMUS and ACDC datasets (update paths in configuration cell below)
- Sufficient disk space (~10GB for processed data)
- GPU recommended for training (classification CV may take several hours)

### 📋 **What This Pipeline Does**
1. **Data Processing**: Converts raw datasets to standardized formats
2. **Cross-Validation**: Trains and evaluates deep learning models
3. **Ablation Studies**: Compares different feature sets (EF-only, volumes-only, all features)
4. **Segmentation**: Evaluates cardiac chamber segmentation on both datasets
5. **Results Analysis**: Aggregates and summarizes all experimental results

### 🎯 **Expected Results**
- **Classification**: ~94% accuracy with all features, ~75% with EF only, ~90% with volumes only
- **Segmentation**: ~95% Dice for CAMUS, ~71% Dice for ACDC
- **Duration**: Full pipeline execution takes 4-8 hours depending on hardware

---

In [ ]:
# Check environment
import sys, torch
print("Python:", sys.version)
print("Torch:", torch.__version__, "| CUDA:", torch.version.cuda, "| available:", torch.cuda.is_available())

In [ ]:
# Configure raw dataset locations
# Update these paths to match your data location
RAW_CAMUS = "cardio_data/raw/camus"  
RAW_ACDC  = "cardio_data/raw/acdc"   
SEED = 42

# Validate dataset availability
print("📁 Dataset Configuration:")
print(f"  CAMUS: {RAW_CAMUS} {'✓' if Path(RAW_CAMUS).exists() else '✗ (not found)'}")
print(f"  ACDC:  {RAW_ACDC} {'✓' if Path(RAW_ACDC).exists() else '✗ (not found)'}")
print(f"  Seed:  {SEED}")

# Check processed data status
processed_camus = Path("cardio_data/processed/camus")
processed_acdc = Path("cardio_data/processed/acdc")
print(f"\n💾 Processed Data Status:")
print(f"  CAMUS: {'✓ Ready' if processed_camus.exists() else '⚠️  Needs processing'}")
print(f"  ACDC:  {'✓ Ready' if processed_acdc.exists() else '⚠️  Needs processing'}")

if not Path(RAW_CAMUS).exists() or not Path(RAW_ACDC).exists():
    print("\n⚠️  WARNING: Some raw datasets not found. Update paths above or skip processing steps.")

In [ ]:
# Process CAMUS (2D ED/ES)
import subprocess, sys
processed_camus = Path("cardio_data/processed/camus")
if processed_camus.exists() and len(list(processed_camus.rglob("*.png"))) > 0:
    print(f"✓ CAMUS data already processed ({len(list(processed_camus.rglob('*.png')))} files found)")
else:
    print("📊 Processing CAMUS dataset...")
    cmd = [sys.executable, "scripts/camus_process.py", "--raw", RAW_CAMUS, "--out", "cardio_data/processed/camus", "--size", "256"]
    print("Running:", " ".join(cmd))
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode == 0:
        print("✓ CAMUS processing completed successfully")
    else:
        print(f"❌ CAMUS processing failed: {result.stderr}")

In [ ]:
# Process ACDC (3D ED/ES)
import subprocess, sys
processed_acdc = Path("cardio_data/processed/acdc")
if processed_acdc.exists() and len(list(processed_acdc.rglob("*.nii.gz"))) > 0:
    print(f"✓ ACDC data already processed ({len(list(processed_acdc.rglob('*.nii.gz')))} files found)")
else:
    print("🧠 Processing ACDC dataset...")
    cmd = [sys.executable, "scripts/acdc_process.py", "--raw", RAW_ACDC, "--out", "cardio_data/processed/acdc", "--target_spacing", "1.25", "1.25", "10.0"]
    print("Running:", " ".join(cmd))
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode == 0:
        print("✓ ACDC processing completed successfully")
    else:
        print(f"❌ ACDC processing failed: {result.stderr}")

In [ ]:
# Make splits
import subprocess, sys
if META.exists():
    print("✓ Metadata file already exists, splits likely already created")
else:
    print("📑 Creating dataset splits...")
    cmd = [sys.executable, "scripts/make_splits.py", "--meta", "meta/master_metadata.csv", "--seed", str(SEED)]
    print("Running:", " ".join(cmd))
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode == 0:
        print("✓ Dataset splits created successfully")
    else:
        print(f"❌ Split creation failed: {result.stderr}")

In [ ]:
# Quick sanity check of metadata
import pandas as pd
df = pd.read_csv("meta/master_metadata.csv")
print(df.head())
print("\nCounts by dataset & split:")
print(df.groupby(["dataset","split"]).size())

In [ ]:
# Cross-validation with hyperparameter optimization
import subprocess, sys
print("🔬 Starting cross-validation with hyperparameter optimization...")
print("Configuration:")
print(f"  - Dataset: CAMUS 4CH ED")
print(f"  - Folds: 5")
print(f"  - HPO trials: 25")
print(f"  - Seed: {SEED}")

cmd = [sys.executable, "scripts/torch_cv.py", "--meta", "meta/master_metadata.csv", "--view", "4CH", "--phase", "ED", "--folds", "5", "--seed", str(SEED), "--trials", "25"]
print("\nRunning:", " ".join(cmd))
print("⚠️  This may take a long time (several hours)...\n")
result = subprocess.run(cmd)
if result.returncode == 0:
    print("✓ Cross-validation completed successfully")
else:
    print(f"❌ Cross-validation failed with return code: {result.returncode}")

In [ ]:
# Segmentation CV (CAMUS) 
import subprocess, sys
print("🖼️ Starting CAMUS segmentation cross-validation...")
print("Configuration:")
print(f"  - Dataset: CAMUS 4CH ED")  
print(f"  - Folds: 5")
print(f"  - Seed: {SEED}")

cmd = [sys.executable, "scripts/seg_cv.py", "--dataset", "camus", "--meta", "meta/master_metadata.csv", "--view", "4CH", "--phase", "ED", "--folds", "5", "--seed", str(SEED)]
print("\nRunning:", " ".join(cmd))
print("⚠️  This may take some time...")
result = subprocess.run(cmd)
if result.returncode == 0:
    print("✓ CAMUS segmentation CV completed successfully")
else:
    print(f"❌ CAMUS segmentation CV failed with return code: {result.returncode}")

In [ ]:
# Segmentation CV (ACDC)
import subprocess, sys
print("🧠 Starting ACDC segmentation cross-validation...")
print("Configuration:")
print(f"  - Dataset: ACDC SAX ED")
print(f"  - Folds: 5") 
print(f"  - Multiclass: True")
print(f"  - Seed: {SEED}")

cmd = [sys.executable, "scripts/seg_cv.py", "--dataset", "acdc", "--meta", "meta/master_metadata.csv", "--view", "SAX", "--phase", "ED", "--folds", "5", "--seed", str(SEED), "--acdc-multiclass"]
print("\nRunning:", " ".join(cmd))
print("⚠️  This may take some time...")
result = subprocess.run(cmd)
if result.returncode == 0:
    print("✓ ACDC segmentation CV completed successfully")
else:
    print(f"❌ ACDC segmentation CV failed with return code: {result.returncode}")

In [ ]:
# Feature extraction and classification ablation studies
import subprocess, sys

# EF-only classification
print("📊 Running EF-only classification...")
cmd = [sys.executable, "scripts/clf_cv.py", "--meta", "meta/master_metadata.csv", "--features", "ef", "--logdir", "logs_ef", "--folds", "5", "--seed", str(SEED)]
print("Running:", " ".join(cmd))
result = subprocess.run(cmd)
if result.returncode == 0:
    print("✓ EF-only classification completed")
else:
    print(f"❌ EF-only classification failed: {result.returncode}")

# Volume-only classification  
print("\n📏 Running volume-only classification...")
cmd = [sys.executable, "scripts/clf_cv.py", "--meta", "meta/master_metadata.csv", "--features", "vol", "--logdir", "logs_vol", "--folds", "5", "--seed", str(SEED)]
print("Running:", " ".join(cmd))
result = subprocess.run(cmd)
if result.returncode == 0:
    print("✓ Volume-only classification completed")
else:
    print(f"❌ Volume-only classification failed: {result.returncode}")

In [ ]:
# Generate comprehensive reports
print("📋 Generating analysis reports...")

# Load and display classification results
clf_results = load_clf_summaries()
if clf_results:
    print("\\n🎯 Classification Results Summary:")
    for setting, df in clf_results.items():
        if not df.empty:
            best_model = df.loc[df['acc_mean'].idxmax()]
            print(f"  {setting.upper()}: {best_model['model']} - {best_model['acc_mean']:.3f} ± {best_model['acc_std']:.3f} accuracy")
else:
    print("⚠️  No classification results found")

# Load and display segmentation results  
seg_results = load_seg_summaries()
if seg_results:
    print("\\n🖼️ Segmentation Results Summary:")
    for name, data in seg_results.items():
        dataset = data.get('dataset', 'unknown')
        dice_mean = data.get('Dice_mean', 0)
        dice_std = data.get('Dice_std', 0)
        print(f"  {dataset.upper()}: {dice_mean:.3f} ± {dice_std:.3f} Dice score")
else:
    print("⚠️  No segmentation results found")

print("\\n✓ Pipeline execution summary complete!")

## Pipeline Summary

This notebook executes the complete cardiac early detection pipeline:

### 🔄 **Data Processing**
1. **CAMUS Processing**: Converts raw echocardiography to 256x256 PNG images
2. **ACDC Processing**: Resamples MRI volumes to 1.25×1.25×10mm spacing
3. **Metadata Generation**: Creates unified dataset splits and metadata

### 🤖 **Model Training & Evaluation**  
4. **Classification CV**: Deep learning model with hyperparameter optimization (25 trials, 5 folds)
5. **CAMUS Segmentation CV**: 2D U-Net for chamber segmentation 
6. **ACDC Segmentation CV**: 3D U-Net for multiclass cardiac structure segmentation
7. **Feature Ablation**: Classification using EF-only vs volume-only features

### 📊 **Analysis & Reports**
8. **Results Aggregation**: Loads and summarizes all experimental results
9. **Performance Comparison**: Compares different approaches and feature sets

Each step includes validation and error checking to ensure robust execution.

In [ ]:
# Pipeline Status Check
print("🔍 Complete Pipeline Status Check")
print("=" * 50)

# Check data processing
processed_camus = Path("cardio_data/processed/camus") 
processed_acdc = Path("cardio_data/processed/acdc")
meta_exists = META.exists()

print("📁 Data Processing:")
print(f"  CAMUS processed: {'✓' if processed_camus.exists() else '✗'}")
print(f"  ACDC processed:  {'✓' if processed_acdc.exists() else '✗'}")
print(f"  Metadata created: {'✓' if meta_exists else '✗'}")

# Check training results
log_dirs = ["logs", "logs_ef", "logs_vol"]
training_status = {}
for log_dir in log_dirs:
    log_path = Path(log_dir)
    has_cls = (log_path / "cv_cls_summary.csv").exists()
    has_seg_camus = (log_path / "cv_seg_camus_summary.json").exists()
    has_seg_acdc = (log_path / "cv_seg_acdc_summary.json").exists()
    training_status[log_dir] = {"cls": has_cls, "seg_camus": has_seg_camus, "seg_acdc": has_seg_acdc}

print(f"\n🤖 Training Results:")
for log_dir, status in training_status.items():
    setting = {"logs": "All Features", "logs_ef": "EF Only", "logs_vol": "Volumes Only"}[log_dir]
    cls_status = "✓" if status["cls"] else "✗"
    seg_status = "✓" if status["seg_camus"] or status["seg_acdc"] else "✗"
    print(f"  {setting}: Classification {cls_status}, Segmentation {seg_status}")

# Overall completion
data_ready = processed_camus.exists() and processed_acdc.exists() and meta_exists
results_ready = any(s["cls"] for s in training_status.values())

print(f"\n📊 Overall Status:")
print(f"  Data Pipeline: {'✓ Complete' if data_ready else '⚠️ Incomplete'}")
print(f"  Training Pipeline: {'✓ Complete' if results_ready else '⚠️ Incomplete'}")

if data_ready and results_ready:
    print("\n🎉 Pipeline fully executed! Ready for analysis.")
elif data_ready:
    print("\n⏳ Data ready. Run training cells to complete pipeline.")
else:
    print("\n🔄 Run data processing cells first, then training cells.")